# Object Lifetime, Memory, and Garbage Collection
This notebook has been rewritten to be slower, more reflective, and less template-like. Instead of treating **Object Lifetime, Memory, and Garbage Collection** as a small isolated topic, the goal here is to treat it as part of Python's larger runtime story: objects are created, names are bound, frames execute bytecode, and the interpreter keeps following rules whether or not the source code looks simple from the outside.

If this topic ever felt a little too neat or too magical when explained quickly, that is exactly what this rewrite is trying to fix. We are going to keep asking two grounding questions: what changed in memory, and what instructions is the interpreter actually stepping through under the surface?

The point is not to drown in internals. The point is to make the language feel explainable. Once that happens, even advanced behavior starts feeling much less mysterious.


When people struggle with **Object Lifetime, Memory, and Garbage Collection**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


- Understand object lifetime in practical terms
- Know what reference counting does
- See why cycles require extra help
- Separate memory cleanup from resource cleanup


An object stays alive as long as something still refers to it. Those references may come from names, containers, object attributes, frames, closures, or global structures. Lifetime is really about reachability.


In [1]:
import sys
value = []
print(sys.getrefcount(value))
other = value
print(sys.getrefcount(value))
del other
print(sys.getrefcount(value))


2
3
2


At the bytecode level you can often see the creation of objects, the storing of references, and the return path. It will not show garbage collection directly, but it does show how names receive references.


In [2]:
import dis

def life():
    data = [1, 2, 3]
    other = data
    return other

dis.dis(life)


  3           0 RESUME                   0

  4           2 BUILD_LIST               0
              4 LOAD_CONST               1 ((1, 2, 3))
              6 LIST_EXTEND              1
              8 STORE_FAST               0 (data)

  5          10 LOAD_FAST                0 (data)
             12 STORE_FAST               1 (other)

  6          14 LOAD_FAST                1 (other)
             16 RETURN_VALUE


When the reference count drops to zero, the object can often be cleaned up immediately.

If two unreachable objects point to each other, their reference counts may never reach zero by themselves.

This is why Python has both ideas in play.

A file descriptor or network connection should be managed explicitly, often with a context manager.


These two objects keep each other alive until the garbage collector notices the cycle is unreachable.


In [3]:
import gc

class Node:
    def __init__(self, name):
        self.name = name
        self.other = None

a = Node("a")
b = Node("b")
a.other = b
b.other = a
print(gc.collect())


34


The inner function still holds onto the captured list even after the outer function finishes.


In [4]:
def factory():
    payload = [1, 2, 3]
    def inner():
        return payload
    return inner

fn = factory()
print(fn())


[1, 2, 3]


The `with` statement is often the simplest answer for resource lifetime.


In [5]:
with open("gc_demo.txt", "w", encoding="utf-8") as f:
    f.write("safe cleanup")
print("done")


done


This is not only a classroom topic. **Object Lifetime, Memory, and Garbage Collection** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


- Using “garbage collection” as a hand-wavy answer to every lifetime question
- Confusing object lifetime with resource lifetime
- Ignoring cycles when teaching object references


- How does CPython usually manage object memory?
- Why are cyclic references special?
- Why is `with open(...)` still useful if Python has garbage collection?


- Reachability controls lifetime.
- Reference counting is common in CPython.
- Cycles need extra collection.
- Resources should still be managed explicitly.


If this notebook did its job, **Object Lifetime, Memory, and Garbage Collection** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.
